# Bulk RNA-seq TME Deconvolution Template

用于 bulk RNA-seq 表达矩阵的免疫/基质评分、细胞比例估计、组间比较和 signature 关联分析。

建议输入 normalized expression，例如 TPM、log2(TPM+1)、或 VST 矩阵；ESTIMATE / CIBERSORT / EPIC 类工具通常需要非 log TPM。运行前请先完成 `RNAseq_General.ipynb`，以生成 `1-DEG/vsd_matrix.csv` 和 `1-DEG/colData.csv`。

## 1. Parameter Configuration

In [ ]:
# ===================== Parameter Configuration =====================
EXPR_FILE <- "./1-DEG/vsd_matrix.csv"     # genes x samples; first column is gene name or row names
META_FILE <- "./1-DEG/colData.csv"         # exported by RNAseq_General; must contain sample and group columns
EXPR_IS_LOG <- TRUE                         # TRUE for log2(TPM+1)/VST; FALSE for TPM-like input
GENE_COLUMN <- NULL                         # NULL = use rownames/first column; or e.g. "gene"
SAMPLE_COLUMN <- "sample"
GROUP_COLUMN <- "condition"
GROUP_LEVELS <- NULL                         # NULL = infer from metadata order; or set c("Control", "Treatment")

# Species of the input expression data. TME methods use human signatures, so mouse
# data are automatically converted from MGI symbols to HGNC symbols via biomaRt.
SPECIES <- "human"                          # "human" or "mouse"

# Optional custom group colors. If NULL, make_group_colors(GROUP_LEVELS) is used.
# Example: GROUP_COLORS <- c("Control" = "#6F6F6F", "Treatment" = "#E07B54")
GROUP_COLORS <- NULL

# ESTIMATE (native implementation)
RUN_ESTIMATE <- TRUE

# IOBR multi-algorithm deconvolution
RUN_IOBR <- TRUE                            # Requires IOBR package
IOBR_METHODS <- c("estimate", "cibersort", "epic", "xcell")
IOBR_PERM <- 1000
IOBR_ARRAYS <- FALSE

# Native CIBERSORT (optional, requires CIBERSORT.R + LM22.txt)
RUN_CIBERSORT <- FALSE
CIBERSORT_SCRIPT <- "./CIBERSORT.R"
CIBERSORT_SIGNATURE <- "./LM22.txt"

# Output
OUTDIR <- "RNAseq_TME_Deconvolution_Output"
dir.create(OUTDIR, showWarnings = FALSE, recursive = TRUE)


## 2. Environment

In [ ]:
options(stringsAsFactors = FALSE)
# First run if needed:
# install.packages(c("tidyverse", "pheatmap", "ggpubr", "corrplot", "RColorBrewer"))
# if (!require("BiocManager", quietly = TRUE)) install.packages("BiocManager")
# BiocManager::install(c("GSVA", "limma"))
# install.packages("estimate", repos = "http://r-forge.r-project.org")
# remotes::install_github("IOBR/IOBR")  # for IOBR-based deconvolution

suppressPackageStartupMessages({
  library(tidyverse)
  library(pheatmap)
  library(ggpubr)
  library(corrplot)
  library(GSVA)
  library(limma)
})

LIB_DIR <- if (dir.exists("RNAseq_lib")) "RNAseq_lib" else "../RNAseq_lib"
source(file.path(LIB_DIR, "plot_utils.R"))
source(file.path(LIB_DIR, "tme_utils.R"))
theme_set(theme_publication())
cat("RNAseq_lib:", LIB_DIR, "\n")


## 3. Load Expression and Metadata

In [ ]:
expr_raw <- read.csv(EXPR_FILE, check.names = FALSE)
if (!is.null(GENE_COLUMN) && GENE_COLUMN %in% colnames(expr_raw)) {
  genes <- expr_raw[[GENE_COLUMN]]
  expr <- as.matrix(expr_raw[, setdiff(colnames(expr_raw), GENE_COLUMN), drop = FALSE])
  rownames(expr) <- genes
} else if (!is.numeric(expr_raw[[1]])) {
  genes <- expr_raw[[1]]
  expr <- as.matrix(expr_raw[, -1, drop = FALSE])
  rownames(expr) <- genes
} else {
  expr <- as.matrix(expr_raw)
}
mode(expr) <- "numeric"
expr <- expr[!duplicated(rownames(expr)) & !is.na(rownames(expr)) & rownames(expr) != "", , drop = FALSE]

meta <- read.csv(META_FILE, check.names = FALSE)

# colData.csv exported by RNAseq_General.ipynb may contain a duplicated 'sample'
# column. Rename duplicates so that meta[[SAMPLE_COLUMN]] behaves as a plain
# vector and downstream subsetting works reliably.
if (sum(colnames(meta) == SAMPLE_COLUMN) > 1) {
  dup_idx <- which(colnames(meta) == SAMPLE_COLUMN)
  colnames(meta)[dup_idx[-1]] <- paste0(SAMPLE_COLUMN, "_", seq_len(length(dup_idx) - 1))
}

if (is.null(GROUP_LEVELS)) GROUP_LEVELS <- unique(as.character(meta[[GROUP_COLUMN]]))
meta[[GROUP_COLUMN]] <- factor(meta[[GROUP_COLUMN]], levels = GROUP_LEVELS)
common_samples <- intersect(colnames(expr), meta[[SAMPLE_COLUMN]])
expr <- expr[, common_samples, drop = FALSE]
meta <- meta[match(common_samples, meta[[SAMPLE_COLUMN]]), ]
stopifnot(all(colnames(expr) == meta[[SAMPLE_COLUMN]]))
cat("Expression:", nrow(expr), "genes x", ncol(expr), "samples\n")
print(table(meta[[GROUP_COLUMN]], useNA = "ifany"))

# Prepare expression for TME deconvolution:
# - reverse log transformation if EXPR_IS_LOG = TRUE
# - convert mouse MGI symbols to HGNC symbols when SPECIES = "mouse"
expr_tme <- prepare_tme_expression(expr, is_log = EXPR_IS_LOG, species = SPECIES)
cat("TME expression matrix:", nrow(expr_tme), "genes x", ncol(expr_tme), "samples\n")

# Resolve group colors for consistent plotting
if (is.null(GROUP_COLORS) || !all(GROUP_LEVELS %in% names(GROUP_COLORS))) {
  group_colors <- make_group_colors(GROUP_LEVELS)
} else {
  group_colors <- GROUP_COLORS[GROUP_LEVELS]
}


## 4. ESTIMATE Score (native implementation)

In [ ]:
if (RUN_ESTIMATE) {
  if (!requireNamespace("estimate", quietly = TRUE)) {
    message("Package 'estimate' is not installed; skipping ESTIMATE.")
  } else {
    estimate_df <- data.frame(NAME = rownames(expr_tme), Description = NA, expr_tme, check.names = FALSE)
    write.table(estimate_df, file.path(OUTDIR, "estimate_input.gct"), sep = "\t", quote = FALSE, row.names = FALSE)
    estimate::filterCommonGenes(input.f = file.path(OUTDIR, "estimate_input.gct"),
                                output.f = file.path(OUTDIR, "estimate_common_genes.gct"), id = "GeneSymbol")
    estimate::estimateScore(file.path(OUTDIR, "estimate_common_genes.gct"),
                            file.path(OUTDIR, "estimate_scores.gct"), platform = "illumina")
    estimate_scores <- read.table(file.path(OUTDIR, "estimate_scores.gct"), skip = 2, header = TRUE, sep = "\t", check.names = FALSE)
    rownames(estimate_scores) <- estimate_scores$NAME
    estimate_scores <- as.data.frame(t(estimate_scores[, -(1:2)]))
    estimate_scores[[SAMPLE_COLUMN]] <- rownames(estimate_scores)
    write.csv(estimate_scores, file.path(OUTDIR, "ESTIMATE_scores.csv"), row.names = FALSE)
    cat("ESTIMATE scores saved to", file.path(OUTDIR, "ESTIMATE_scores.csv"), "\n")
  }
}


## 5. IOBR Multi-algorithm TME Deconvolution

In [ ]:
iobr_results <- list()
if (RUN_IOBR) {
  if (!requireNamespace("IOBR", quietly = TRUE)) {
    message("Package 'IOBR' is not installed; skipping IOBR deconvolution.")
  } else {
    iobr_results <- run_iobr_deconvolution(
      expr_tme,
      methods = IOBR_METHODS,
      perm = IOBR_PERM,
      arrays = IOBR_ARRAYS,
      id_column = SAMPLE_COLUMN
    )
    # Save individual results
    for (method in names(iobr_results)) {
      write.csv(iobr_results[[method]], file.path(OUTDIR, paste0("IOBR_", method, ".csv")), row.names = FALSE)
    }
    # Combine all results
    if (length(iobr_results) >= 2) {
      tme_combined <- combine_tme_results(iobr_results, id_column = SAMPLE_COLUMN)
      write.csv(tme_combined, file.path(OUTDIR, "IOBR_TME_combined.csv"), row.names = FALSE)
      cat("Combined TME table:", nrow(tme_combined), "samples x", ncol(tme_combined), "features\n")
    }
  }
}


## 6. Native CIBERSORT (optional)

In [ ]:
if (RUN_CIBERSORT) {
  if (!file.exists(CIBERSORT_SCRIPT) || !file.exists(CIBERSORT_SIGNATURE)) {
    stop("CIBERSORT script/signature file not found. Check CIBERSORT_SCRIPT and CIBERSORT_SIGNATURE.")
  }
  source(CIBERSORT_SCRIPT)
  write.table(data.frame(GeneSymbol = rownames(expr_tme), expr_tme, check.names = FALSE),
              file.path(OUTDIR, "CIBERSORT_input.txt"), sep = "\t", quote = FALSE, row.names = FALSE)
  cib_res <- CIBERSORT(CIBERSORT_SIGNATURE, file.path(OUTDIR, "CIBERSORT_input.txt"), perm = 100, QN = TRUE)
  write.csv(cib_res, file.path(OUTDIR, "CIBERSORT_results.csv"))
}


## 7. TME Visualization

In [ ]:
group_df_for_plot <- meta[, c(SAMPLE_COLUMN, GROUP_COLUMN), drop = FALSE]

# ESTIMATE scores boxplot (from IOBR if available, otherwise native)
if (RUN_IOBR && "estimate" %in% names(iobr_results)) {
  est_long <- melt_estimate_scores(iobr_results[["estimate"]], id_column = SAMPLE_COLUMN,
                                   group_df = group_df_for_plot,
                                   sample_col = SAMPLE_COLUMN, group_col = GROUP_COLUMN)
  plot_estimate_boxplot_pdf(
    est_long,
    group_col = GROUP_COLUMN,
    filename = file.path(OUTDIR, "IOBR_ESTIMATE_scores_boxplot.pdf"),
    title = "ESTIMATE Scores by Group",
    group_colors = group_colors
  )

  # IOBR ESTIMATE score heatmap
  plot_tme_heatmap_pdf(
    iobr_results[["estimate"]], meta,
    group_col = GROUP_COLUMN, sample_col = SAMPLE_COLUMN,
    group_colors = group_colors,
    filename = file.path(OUTDIR, "IOBR_ESTIMATE_heatmap.pdf"),
    title = "IOBR ESTIMATE Scores",
    width = 8, height = 6
  )
}

# CIBERSORT stacked barplot + per-cell-type boxplot + individual celltype plots
if (RUN_IOBR && "cibersort" %in% names(iobr_results)) {
  cib_long <- melt_tme_results(iobr_results[["cibersort"]], id_column = SAMPLE_COLUMN,
                               group_df = group_df_for_plot,
                               sample_col = SAMPLE_COLUMN, group_col = GROUP_COLUMN)
  # Exclude P-value and correlation columns from plotting
  cib_long <- cib_long |> dplyr::filter(!grepl("P-value|Correlation|RMSE", .data$cell_type))

  cib_bar_size <- calc_tme_barplot_size(n_samples = length(unique(cib_long[[SAMPLE_COLUMN]])),
                                       n_celltypes = length(unique(cib_long$cell_type)))
  plot_tme_barplot_pdf(cib_long, group_col = GROUP_COLUMN, sample_col = SAMPLE_COLUMN,
                       filename = file.path(OUTDIR, "IOBR_CIBERSORT_barplot.pdf"),
                       title = "CIBERSORT Cell Fractions",
                       width = cib_bar_size["width"], height = cib_bar_size["height"])

  cib_box_size <- calc_tme_boxplot_size(n_celltypes = length(unique(cib_long$cell_type)))
  plot_tme_boxplot_pdf(cib_long, group_col = GROUP_COLUMN, value_col = "fraction",
                       filename = file.path(OUTDIR, "IOBR_CIBERSORT_boxplot.pdf"),
                       title = "CIBERSORT Cell Fraction by Group",
                       width = cib_box_size["width"], height = cib_box_size["height"],
                       group_colors = group_colors)

  plot_tme_per_celltype_pdf(
    cib_long,
    group_col = GROUP_COLUMN, value_col = "fraction",
    filename_prefix = file.path(OUTDIR, "IOBR_CIBERSORT"),
    title_prefix = "CIBERSORT",
    group_colors = group_colors
  )
}

# EPIC stacked barplot + per-cell-type boxplot + individual celltype plots
if (RUN_IOBR && "epic" %in% names(iobr_results)) {
  epic_long <- melt_tme_results(iobr_results[["epic"]], id_column = SAMPLE_COLUMN,
                                group_df = group_df_for_plot,
                                sample_col = SAMPLE_COLUMN, group_col = GROUP_COLUMN)

  epic_bar_size <- calc_tme_barplot_size(n_samples = length(unique(epic_long[[SAMPLE_COLUMN]])),
                                        n_celltypes = length(unique(epic_long$cell_type)))
  plot_tme_barplot_pdf(epic_long, group_col = GROUP_COLUMN, sample_col = SAMPLE_COLUMN,
                       filename = file.path(OUTDIR, "IOBR_EPIC_barplot.pdf"),
                       title = "EPIC Cell Fractions",
                       width = epic_bar_size["width"], height = epic_bar_size["height"])

  epic_box_size <- calc_tme_boxplot_size(n_celltypes = length(unique(epic_long$cell_type)))
  plot_tme_boxplot_pdf(epic_long, group_col = GROUP_COLUMN, value_col = "fraction",
                       filename = file.path(OUTDIR, "IOBR_EPIC_boxplot.pdf"),
                       title = "EPIC Cell Fraction by Group",
                       width = epic_box_size["width"], height = epic_box_size["height"],
                       group_colors = group_colors)

  plot_tme_per_celltype_pdf(
    epic_long,
    group_col = GROUP_COLUMN, value_col = "fraction",
    filename_prefix = file.path(OUTDIR, "IOBR_EPIC"),
    title_prefix = "EPIC",
    group_colors = group_colors
  )
}

# xCell score heatmap
if (RUN_IOBR && "xcell" %in% names(iobr_results)) {
  plot_tme_heatmap_pdf(
    iobr_results[["xcell"]], meta,
    group_col = GROUP_COLUMN, sample_col = SAMPLE_COLUMN,
    group_colors = group_colors,
    filename = file.path(OUTDIR, "IOBR_xCell_heatmap.pdf"),
    title = "xCell Scores",
    width = 10, height = 12
  )
}


## 8. ssGSEA Immune Signature Scoring

In [ ]:
immune_gene_sets <- list(
  T_cell = c("CD3D", "CD3E", "CD2", "TRAC"),
  CD8_T_cell = c("CD8A", "CD8B", "GZMB", "PRF1"),
  NK_cell = c("NKG7", "GNLY", "KLRD1", "KLRK1"),
  B_cell = c("MS4A1", "CD79A", "CD79B", "CD19"),
  Myeloid = c("LYZ", "S100A8", "S100A9", "FCGR3A"),
  Macrophage = c("CD68", "C1QA", "C1QB", "CSF1R"),
  CAF = c("COL1A1", "COL1A2", "ACTA2", "FAP"),
  Endothelial = c("PECAM1", "VWF", "KDR", "ENG")
)

# The bundled immune signatures are human gene symbols; convert mouse rownames if needed.
expr_for_ssgsea <- if (SPECIES == "mouse") {
  convert_mouse_symbols_to_human(expr, verbose = FALSE)
} else {
  as.data.frame(expr, check.names = FALSE)
}

row_upper <- toupper(rownames(expr_for_ssgsea))
upper_to_real <- setNames(rownames(expr_for_ssgsea), row_upper)
gs <- lapply(immune_gene_sets, function(x) unique(upper_to_real[intersect(toupper(x), names(upper_to_real))]))
gs <- gs[lengths(gs) >= 2]

params <- gsvaParam(as.matrix(expr_for_ssgsea), gs, kcdf = "Gaussian", minSize = 2, maxSize = Inf)
ssgsea_scores <- gsva(params, verbose = FALSE)
write.csv(ssgsea_scores, file.path(OUTDIR, "ssGSEA_immune_scores.csv"))


## 9. ssGSEA Group Comparison and Heatmap

In [ ]:
score_df <- as.data.frame(t(ssgsea_scores)) %>% rownames_to_column(SAMPLE_COLUMN) %>% left_join(meta, by = SAMPLE_COLUMN)
score_long <- score_df %>% pivot_longer(cols = names(gs), names_to = "signature", values_to = "score")

score_long[[GROUP_COLUMN]] <- factor(score_long[[GROUP_COLUMN]], levels = GROUP_LEVELS)
score_comparisons <- if (length(GROUP_LEVELS) >= 2) combn(GROUP_LEVELS, 2, simplify = FALSE) else list()
p_box <- plot_group_boxplot_pdf(
  score_long,
  value_col = "score",
  group_col = GROUP_COLUMN,
  facet_col = "signature",
  comparisons = score_comparisons,
  method = "t.test",
  title = "ssGSEA Immune Signature Scores",
  ylab = "ssGSEA score",
  group_colors = group_colors,
  filename = file.path(OUTDIR, "ssGSEA_group_boxplot.pdf"),
  width = 12,
  height = 8
)
print(p_box)

ann <- data.frame(
  Group = as.character(meta[[GROUP_COLUMN]])
)
rownames(ann) <- meta[[SAMPLE_COLUMN]]
ann$Group <- factor(ann$Group, levels = GROUP_LEVELS)
annotation_colors <- list(Group = group_colors)
pheatmap(ssgsea_scores, annotation_col = ann, annotation_colors = annotation_colors,
         scale = "row", filename = file.path(OUTDIR, "ssGSEA_heatmap.pdf"), width = 8, height = 7)
writeLines(capture.output(sessionInfo()), file.path(OUTDIR, "sessionInfo.txt"))
